In [0]:
%sql
CREATE TABLE IF NOT EXISTS mortgage_catalog.metadata.audit_log
(
    run_id              STRING,
    table_name          STRING,
    bronze_count        BIGINT,
    silver_count        BIGINT,
    quarantine_count    BIGINT,
    rejected_count      BIGINT,
    status              STRING,
    audit_timestamp     TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import current_timestamp
import uuid

# ==========================================
# 08 - GENERIC AUDIT & RECONCILIATION
# ==========================================

run_id = str(uuid.uuid4())

configs = (
    spark.table("mortgage_catalog.metadata.dq_rules")
    .filter("is_active = true")
    .select("table_name")
    .distinct()
    .collect()
)

audit_records = []

for config in configs:

    bronze_table = config["table_name"]
    entity = bronze_table.replace("_bronze", "")

    bronze = f"mortgage_catalog.bronze.{bronze_table}"
    silver = f"mortgage_catalog.silver.{entity}_silver"
    quarantine = f"mortgage_catalog.bronze.{entity}_quarantine"

    # ----------------------------------
    # Counts
    # ----------------------------------

    bronze_count = (
        spark.table(bronze).count()
        if spark.catalog.tableExists(bronze)
        else 0
    )

    silver_count = (
        spark.table(silver).count()
        if spark.catalog.tableExists(silver)
        else 0
    )

    quarantine_count = (
        spark.table(quarantine).count()
        if spark.catalog.tableExists(quarantine)
        else 0
    )

    # ----------------------------------
    # Reconciliation
    # ----------------------------------

    rejected_count = bronze_count - (
        silver_count + quarantine_count
    )

    status = (
        "PASS"
        if rejected_count == 0
        else "FAIL"
    )

    audit_records.append(
        (
            run_id,
            bronze_table,
            bronze_count,
            silver_count,
            quarantine_count,
            rejected_count,
            status
        )
    )

# ==========================================
# WRITE AUDIT
# ==========================================

audit_df = spark.createDataFrame(
    audit_records,
    """
    run_id STRING,
    table_name STRING,
    bronze_count LONG,
    silver_count LONG,
    quarantine_count LONG,
    rejected_count LONG,
    status STRING
    """
)

audit_df = audit_df.withColumn(
    "audit_timestamp",
    current_timestamp()
)

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "mortgage_catalog.metadata.audit_log"
    )
)

display(audit_df)

In [0]:
%sql
SELECT *
FROM mortgage_catalog.metadata.audit_log
ORDER BY audit_timestamp DESC;

run_id,table_name,bronze_count,silver_count,quarantine_count,rejected_count,status,audit_timestamp
28082a45-a0af-4b47-b19d-ad679353c44c,document_bronze,3,3,0,0,PASS,2026-10-02T14:54:47.286Z
28082a45-a0af-4b47-b19d-ad679353c44c,payment_bronze,5,5,0,0,PASS,2026-10-02T14:54:47.286Z
28082a45-a0af-4b47-b19d-ad679353c44c,credit_score_bronze,5,5,0,0,PASS,2026-10-02T14:54:47.286Z
28082a45-a0af-4b47-b19d-ad679353c44c,branch_bronze,3,3,0,0,PASS,2026-10-02T14:54:47.286Z
28082a45-a0af-4b47-b19d-ad679353c44c,loan_application_bronze,5,5,0,0,PASS,2026-10-02T14:54:47.286Z
28082a45-a0af-4b47-b19d-ad679353c44c,property_bronze,3,3,0,0,PASS,2026-10-02T14:54:47.286Z
28082a45-a0af-4b47-b19d-ad679353c44c,customer_autoloader,11,8,3,0,PASS,2026-10-02T14:54:47.286Z
